# Step 8. 🎯 도전 과제
### 기존 toolcalling_Practice_261008에 붙여넣어 사용해주셔야합니다.
본인이 관심 있는 도구를 직접 만들어보세요. **Pydantic 정형 반환 패턴**을 적용하면서.

```python
class MyResult(BaseModel):
    field1: str
    field2: int

@tool
def my_tool(arg: type) -> MyResult:
    """..."""
    return MyResult(field1="...", field2=42)
```

---


## 과제 A: 단위 변환기 (실무 빈도 ★★★★★)

길이·무게·온도 변환 도구. 반환값을 `ConversionResult` Pydantic 객체로.

**힌트**
```python
class ConversionResult(BaseModel):
    original_value: float
    original_unit: str
    converted_value: float
    converted_unit: str

@tool
def convert_length(value: float, from_unit: str, to_unit: str) -> ConversionResult:
    """길이 단위를 변환합니다.

    Args:
        value: 변환할 값
        from_unit: 출발 단위 ('m', 'cm', 'km', 'inch', 'ft')
        to_unit: 도착 단위 ('m', 'cm', 'km', 'inch', 'ft')
    """
    # 변환 로직
    ...
```

테스트: "180cm가 몇 미터야?", "10km는 몇 마일이야?"

In [ ]:
class ConversionResult(BaseModel):
    original_value: float = Field(description="변환할 값")
    original_unit: Literal['m', 'cm', 'km', 'inch', 'ft', 'mile'] = Field(description="출발 단위")
    converted_value: float =Field(description="변환된 값")
    converted_unit: Literal['m', 'cm', 'km', 'inch', 'ft','mile'] = Field(description="도착 단위")

@tool
def convert_length(value: float, from_unit: str, to_unit: str) -> ConversionResult:
    """길이 단위를 변환합니다.

    Args:
        value: 변환할 값
        from_unit: 출발 단위 ('m', 'cm', 'km', 'inch', 'ft', 'mile')
        to_unit: 도착 단위 ('m', 'cm', 'km', 'inch', 'ft', 'mile')
        """
    LENGTH_TO_M = {"m": 1.0, "cm": 0.01, "km": 1000.0, "inch": 0.0254, "ft": 0.3048,"mile": 1609.344}
    
    converted = value*(LENGTH_TO_M[from_unit]/LENGTH_TO_M[to_unit])
    return ConversionResult(original_value=value, 
                            original_unit=from_unit, 
                            converted_value=converted,
                            converted_unit=to_unit )
print("길이 단위 변환 완료")
# ---------------------------------------------
print()
print("Pydantic 객체 출력")
test = convert_length.invoke({"value":100,"from_unit":"cm","to_unit":"m"})
print(f"타입:  {type(test).__name__}")
print(f"객체:  {test}")
print(f"속성:  convert_value={test.converted_value}, from={test.original_unit}, to={test.converted_unit}")
#--------------------------------------------------
print()
# all_tools.append(convert_length)
print("📋 등록된 도구의 반환 타입:")
for t in all_tools:
    print(f"   🔧 {t.name:25s} → {t.func.__annotations__.get('return', '?')}")
print()
que = "180cm가 몇 미터야? 10km는 몇 마일이야?"
result = run_with_tools(que,all_tools)
print(result)

길이 단위 변환 완료

Pydantic 객체 출력
타입:  ConversionResult
객체:  original_value=100.0 original_unit='cm' converted_value=1.0 converted_unit='m'
속성:  convert_value=1.0, from=cm, to=m

📋 등록된 도구의 반환 타입:
   🔧 multiply                  → <class 'float'>
   🔧 get_exchange_rate         → <class '__main__.ExchangeRateResult'>
   🔧 get_current_time          → <class '__main__.TimeInfo'>
   🔧 search_product            → <class '__main__.ProductInfo'>
   🔧 convert_length            → <class '__main__.ConversionResult'>

180cm는 1.8미터입니다.  
10km는 약 6.21마일입니다.


---

## 과제 B: 날짜 계산기 (실무 빈도 ★★★★☆)

```python
class DateDiff(BaseModel):
    date1: str
    date2: str
    days: int
    weeks: int

@tool
def days_between(date1: str, date2: str) -> DateDiff:
    """두 날짜(YYYY-MM-DD) 사이의 일수·주수를 반환합니다."""
    ...
```

In [ ]:
from datetime import datetime
class DateDiff(BaseModel):
    date1: str =Field(description="시작 날짜(YYYY-MM-DD)")
    date2: str =Field(description="종료 날짜(YYYY-MM-DD)")
    days: int =Field(description="두 날짜 사이 일수")
    weeks: int =Field(description="days//7 소수점은 버릴 것")

@tool
def days_between(date1: str, date2: str) -> DateDiff:
    """두 날짜(YYYY-MM-DD) 사이의 일수·주수를 반환합니다.
    
    Args:
        date1: 시작 날짜
        date2: 종료 날짜
    """
    format_string = "%Y-%m-%d"
    d1= datetime.strptime(date1,format_string)
    d2= datetime.strptime(date2,format_string)
    days = abs((d2 - d1).days)
    weeks = days//7

    return DateDiff(date1=date1,
                     date2=date2,
                     days=days,
                     weeks=weeks)
#-----------------------------------
# all_tools.append(days_between)
print("📋 등록된 도구의 반환 타입:")
for t in all_tools:
    print(f"   🔧 {t.name:25s} → {t.func.__annotations__.get('return', '?')}")
print()
# 1단계: 도구만 먼저 확인 (직접 계산하기 쉬운 값)
print(days_between.invoke({"date1": "2026-10-01", "date2": "2026-10-08"}))
print()
# 기대: date1='2026-10-01' date2='2026-10-08' days=7 weeks=1
#-----------------------------------------------------------------

# # 2단계: 순서를 뒤집은 경우
print(days_between.invoke({"date1": "2026-10-08", "date2": "2026-10-01"}))
print()

# # 3단계: LLM과 연결
print(days_between.invoke({"date1": "2026-10-01", "date2": "2026-10-29"}))
print(run_with_tools("2026-10-01부터 2026-10-29까지 며칠이야?", [days_between]))
# # 기대: 28일, 4주

📋 등록된 도구의 반환 타입:
   🔧 multiply                  → <class 'float'>
   🔧 get_exchange_rate         → <class '__main__.ExchangeRateResult'>
   🔧 get_current_time          → <class '__main__.TimeInfo'>
   🔧 search_product            → <class '__main__.ProductInfo'>
   🔧 convert_length            → <class '__main__.ConversionResult'>
   🔧 days_between              → <class '__main__.DateDiff'>

date1='2026-10-01' date2='2026-10-08' days=7 weeks=1

date1='2026-10-08' date2='2026-10-01' days=7 weeks=1

date1='2026-10-01' date2='2026-10-29' days=28 weeks=4
2026년 10월 1일부터 2026년 10월 29일까지는 총 28일입니다.
